In [7]:
from myutils import display_ppm_image,  convert_jfif_to_ppm,predict_traffic_sign_all,ppm_to_tensor_tf
from GTSRB_utils import GTSRB_CLASSES, predict_traffic_sign, create_subset_loader, load_ppm_image
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import load_model
model = load_model("/content/EfficientNetB1.keras")

In [1]:
import os

# Path to the folder
folder_path = "Test_Data"

# Count PNG files
png_count = sum(1 for file in os.listdir(folder_path) if file.lower().endswith(".png"))

print(f"Number of .png files in '{folder_path}': {png_count}")

Number of .png files in 'Test_Data': 516


 **Taking 3 samples from each class as a reduced testset to decrease computation time, so now new Test set have 129 samples instead of 516**

In [2]:
import os
import shutil
import pandas as pd

# Paths
folder_path = "/content/Test_Data"
csv_file_path = "/content/Test_Data/Attack_Test.csv"
new_folder_path = "/content/threesampleperclass"

# Create a new folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)

# Read the CSV file
df = pd.read_csv(csv_file_path)

# Ensure 'ClassId' is treated as integer and 'path' is treated as string
df['ClassId'] = df['ClassId'].astype(int)
df['Path'] = df['Path'].astype(str)

# Iterate over each class and select 3 samples
for label in range(43):  # Assuming class labels are from 0 to 42
    # Filter the samples of the current class
    class_samples = df[df['ClassId'] == label]

    # Get the first 3 samples for this class
    selected_samples = class_samples.head(3)

    # Move the selected samples to the new folder
    for _, row in selected_samples.iterrows():
        file_name = row['Path']
        src_file_path = os.path.join(folder_path, file_name)
        dest_file_path = os.path.join(new_folder_path, file_name)

        # Check if the file exists before copying
        if os.path.exists(src_file_path):
            shutil.copy(src_file_path, dest_file_path)

print("Successfully created the 'threesampleperclass' folder with 3 samples per class.")


Successfully created the 'threesampleperclass' folder with 3 samples per class.


In [3]:
import os
import pandas as pd

# Path to the new folder and CSV file
new_folder_path = "/content/threesampleperclass"
csv_file_path = "/content/Test_Data/Attack_Test.csv"

# Read the CSV file
df = pd.read_csv(csv_file_path)

# Ensure 'ClassId' is treated as integer and 'path' is treated as string
df['ClassId'] = df['ClassId'].astype(int)
df['Path'] = df['Path'].astype(str)

# Create a dictionary to store the count of each class
class_counts = {i: 0 for i in range(43)}

# Iterate over each image in the new directory and count its class
for file_name in os.listdir(new_folder_path):
    # For each file in the new directory, find its corresponding class from the CSV
    file_path = os.path.join(new_folder_path, file_name)

    if os.path.isfile(file_path):
        # Find the class of this image based on its path
        matching_row = df[df['Path'] == file_name]

        if not matching_row.empty:
            class_id = matching_row.iloc[0]['ClassId']
            class_counts[class_id] += 1

# Display the results
print("Class distribution in 'threesampleperclass' directory:")
for class_id, count in class_counts.items():
    print(f"Class {class_id}: {count} samples")

# Calculate the total number of images
total_images = sum(class_counts.values())
print(f"Total number of images in the new directory: {total_images}")


Class distribution in 'threesampleperclass' directory:
Class 0: 3 samples
Class 1: 3 samples
Class 2: 3 samples
Class 3: 3 samples
Class 4: 3 samples
Class 5: 3 samples
Class 6: 3 samples
Class 7: 3 samples
Class 8: 3 samples
Class 9: 3 samples
Class 10: 3 samples
Class 11: 3 samples
Class 12: 3 samples
Class 13: 3 samples
Class 14: 3 samples
Class 15: 3 samples
Class 16: 3 samples
Class 17: 3 samples
Class 18: 3 samples
Class 19: 3 samples
Class 20: 3 samples
Class 21: 3 samples
Class 22: 3 samples
Class 23: 3 samples
Class 24: 3 samples
Class 25: 3 samples
Class 26: 3 samples
Class 27: 3 samples
Class 28: 3 samples
Class 29: 3 samples
Class 30: 3 samples
Class 31: 3 samples
Class 32: 3 samples
Class 33: 3 samples
Class 34: 3 samples
Class 35: 3 samples
Class 36: 3 samples
Class 37: 3 samples
Class 38: 3 samples
Class 39: 3 samples
Class 40: 3 samples
Class 41: 3 samples
Class 42: 3 samples
Total number of images in the new directory: 129


**Running Deepfool attack on The reduced Test set and evaluating The Adverserial Accuracy**

In [5]:
import tensorflow as tf
import numpy as np

def deepfool_attack(
    model,
    img_input,
    num_classes,
    max_iter=20,
    overshoot=0.02,
    eps=1e-6,
    clip_min=0.0,
    clip_max=255.0,
    verbose=True
):
    """
    Perform DeepFool attack on a single image in the [0, 255] range.

    Args:
        model: TF/Keras model (must output logits, not probabilities).
        img_input: Input image tensor with shape (1, H, W, C) in [0, 255].
        num_classes: Number of output classes.
        max_iter: Maximum iterations (default: 50).
        overshoot: Perturbation safety factor (default: 0.02).
        eps: Small value to avoid division by zero.
        clip_min/clip_max: Pixel value bounds (default: 0, 255).
        verbose: Whether to print attack progress (default: True).

    Returns:
        Adversarial image tensor (same shape as img_input).
    """
    # Convert input to tensor and initialize variables
    img_input = tf.convert_to_tensor(img_input, dtype=tf.float32)
    adv_image = tf.Variable(img_input, trainable=True)
    original_image = tf.identity(img_input)
    total_perturbation = tf.Variable(tf.zeros_like(original_image), trainable=True)

    # Get original prediction
    original_logits = model(original_image)
    original_class = tf.argmax(original_logits[0]).numpy()

    if verbose:
        print(f"Initial prediction: Class {original_class}")

    for iteration in range(max_iter):
        with tf.GradientTape() as tape:
            tape.watch(adv_image)
            logits = model(adv_image)

        current_class = tf.argmax(logits[0]).numpy()

        # Debug prints (before checking for success)
        if verbose:
            print(f"\nIteration {iteration}:")
            print(f"  Current class: {current_class}")

        # Check for misclassification
        if current_class != original_class:
            if verbose:
                print(f"\nAttack succeeded at iteration {iteration}!")
                print(f"Original class: {original_class} -> Adversarial class: {current_class}")
            break

        # Compute gradients of all logits w.r.t. input
        grads = tape.jacobian(logits, adv_image)  # Shape: (1, num_classes, 1, H, W, C)
        grads = tf.squeeze(grads, axis=[0, 2])    # Remove batch dim -> (num_classes, H, W, C)

        logits_diff = logits[0] - logits[0, original_class]
        min_perturb_norm = float('inf')
        best_perturb = None

        # Find minimal perturbation across all classes
        for k in range(num_classes):
            if k == original_class:
                continue

            w_k = grads[k] - grads[original_class]  # Shape: (H, W, C)
            f_k = logits_diff[k]

            norm_w_k = tf.norm(tf.reshape(w_k, [-1]))
            if norm_w_k < eps:
                continue

            # Compute perturbation for class k
            perturb_k = (tf.abs(f_k) / (norm_w_k**2 + eps)) * w_k
            perturb_k = tf.expand_dims(perturb_k, axis=0)  # Add batch dim -> (1, H, W, C)

            perturb_k_norm = tf.norm(perturb_k)
            if perturb_k_norm < min_perturb_norm:
                min_perturb_norm = perturb_k_norm
                best_perturb = perturb_k

        if best_perturb is None:
            if verbose:
                print("No valid perturbation found - terminating early")
            break

        # Update perturbation and adversarial image
        total_perturbation.assign_add(best_perturb)
        adv_image.assign(original_image + (1 + overshoot) * total_perturbation)
        adv_image.assign(tf.clip_by_value(adv_image, clip_min, clip_max))

    if verbose and current_class == original_class:
        print("\nAttack failed to misclassify within max iterations")

    return adv_image.numpy()

In [8]:
import os
import tensorflow as tf
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Path to the new directory
new_directory_path = "/content/threesampleperclass"

# Get all image file paths from the new directory
image_filenames = [file for file in os.listdir(new_directory_path) if file.lower().endswith(".png")]

# Assuming the same CSV structure for labels and filenames
df = pd.read_csv("/content/Test_Data/Attack_Test.csv")

# Initialize image height and width for resizing
img_height = 240  # Update according to model input
img_width = 240   # Update according to model input

# Assume model is already defined elsewhere
model =load_model("/content/EfficientNetB1.keras")   # Your pre-trained model here
batch_size = 32



# Turn off verbose output
VERBOSE_ATTACK = False

# Store adversarial samples and labels
adv_images = []
adv_labels = []

# Loop through the clean dataset
for image_path in tqdm(image_filenames, total=len(image_filenames)):
    # Extract the corresponding label from the dataframe based on the image path
    matching_row = df[df['Path'] == image_path]
    if not matching_row.empty:
        label = matching_row.iloc[0]['ClassId']

        # Load and preprocess image
        full_path = os.path.join(new_directory_path, image_path)
        img_raw = tf.io.read_file(full_path)
        img = tf.image.decode_png(img_raw, channels=3)
        img = tf.image.resize(img, [img_height, img_width])
        img = tf.expand_dims(img, axis=0)  # Shape: (1, H, W, C)
        img = tf.cast(img, tf.float32)

        # Apply DeepFool
        adv_img = deepfool_attack(model, img, num_classes=43, verbose=VERBOSE_ATTACK)

        # Collect results
        adv_images.append(adv_img[0])  # Remove batch dim
        adv_labels.append(label)

# Convert to Tensors and batch
adv_images_tensor = tf.convert_to_tensor(adv_images)
adv_labels_tensor = tf.convert_to_tensor(adv_labels)

adv_dataset = tf.data.Dataset.from_tensor_slices((adv_images_tensor, adv_labels_tensor))
adv_dataset = adv_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Evaluate on adversarial dataset
y_true_adv = []
y_pred_adv = []

for batch_images, batch_labels in adv_dataset:
    preds = model.predict(batch_images)
    pred_classes = tf.argmax(preds, axis=1).numpy()
    y_true_adv.extend(batch_labels.numpy())
    y_pred_adv.extend(pred_classes)

# Report results
acc_adv = accuracy_score(y_true_adv, y_pred_adv)
print(f"\nAdversarial Test Accuracy (DeepFool): {acc_adv:.4f}")
print("\nClassification Report (DeepFool):")
print(classification_report(y_true_adv, y_pred_adv))

print("\nConfusion Matrix (DeepFool):")
print(confusion_matrix(y_true_adv, y_pred_adv))


100%|██████████| 129/129 [1:16:26<00:00, 35.56s/it]


1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step

Adversarial Test Accuracy (DeepFool): 0.5271

Classification Report (DeepFool):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.33      0.67      0.44         3
           2       0.67      0.67      0.67         3
           3       0.33      0.33      0.33         3
           4       0.33      0.67      0.44         3
           5       1.00      0.33      0.50         3
           6       0.00      0.00      0.00         3
           7       0.43      1.00      0.60         3
           8       0.50      0.33      0.40         3
           9       1.00      1.00      1.00         3
          10       0.75      1.00      0.86         3
          11       0.33      0.67      0.44         3
          12       0.67   

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Running The  Test Accuracy  on reduced Testset before the Attack**

In [23]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os

# Load CSV and extract paths/labels
labels_df = pd.read_csv("/content/Test_Data/Attack_Test.csv")
image_filenames = labels_df['Path'].values  # e.g., '00000.png'
labels = labels_df['ClassId'].values       # Class IDs (0-42)

# Filter only files that exist in the threesampleperclass directory
base_dir = "/content/threesampleperclass"
existing_files = set(os.listdir(base_dir))
filtered_filenames = []
filtered_labels = []

for fname, label in zip(image_filenames, labels):
    if fname in existing_files:
        filtered_filenames.append(fname)
        filtered_labels.append(label)

# Convert to numpy arrays
filtered_filenames = np.array(filtered_filenames)
filtered_labels = np.array(filtered_labels)

# Preprocessing function (resize to 240x240)
def load_and_preprocess_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)  # RGB
    img = tf.image.resize(img, [240, 240])      # Resize to model input size
    return img

# Create TensorFlow Dataset
def create_dataset(image_paths, labels):
    image_paths = [os.path.join(base_dir, fname) for fname in image_paths]
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(
        lambda path, label: (load_and_preprocess_image(path), label),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.batch(32).prefetch(tf.data.AUTOTUNE)
    return dataset

# Build dataset
reduced_dataset = create_dataset(filtered_filenames, filtered_labels)

# Evaluate
y_true, y_pred = [], []
for batch_images, batch_labels in reduced_dataset:
    preds = model.predict(batch_images, verbose=0)
    pred_classes = tf.argmax(preds, axis=1).numpy()
    y_true.extend(batch_labels.numpy())
    y_pred.extend(pred_classes)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy: 0.9690

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3
           2       1.00      1.00      1.00         3
           3       1.00      1.00      1.00         3
           4       1.00      1.00      1.00         3
           5       1.00      1.00      1.00         3
           6       1.00      1.00      1.00         3
           7       1.00      1.00      1.00         3
           8       1.00      1.00      1.00         3
           9       1.00      1.00      1.00         3
          10       1.00      1.00      1.00         3
          11       1.00      1.00      1.00         3
          12       1.00      1.00      1.00         3
          13       1.00      1.00      1.00         3
          14       1.00      1.00      1.00         3
          15       1.00      1.00      1.00         3
          16       1.00      1.00      1